# Phase 2a-3. TME composition analysis

이 notebook의 목적은 TAM subtype별 세포 수/비율을 sample 단위로 계산하여, 종양 미세환경(TME)에서 C1QC+ TAM과 SPP1+ TAM의 구성 비율이 샘플마다 어떻게 다른지 확인하는 것이다

DEG가 gene-level 분석이라면, TME composition은 cell-level 분석이다

- DEG: 어떤 gene이 subtype을 구분하는가?
- TME composition: 어떤 cell subtype이 sample마다 얼마나 존재하는가?

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

mac = sc.read_h5ad('../phase1_scrna/dataset/GSE127465_mac_phase2a.h5ad')
print(mac)
print(mac.obs['tam_subtype'].value_counts())
print((mac.obs['tam_subtype'].value_counts(normalize=True) * 100).round(2))


## 1. 샘플별 TAM subtype composition 계산

전체 macrophage subset에서 subtype 비율을 보는 것도 중요하지만, 논문/임상적 질문에서는 sample 또는 patient 단위의 차이가 더 중요하다.

예:
- 어떤 sample은 C1QC+ TAM 비율이 높을 수 있음
- 어떤 sample은 SPP1+ TAM 비율이 높을 수 있음
- 어떤 sample은 macrophage subtype annotation이 거의 Unknown으로 남을 수 있음

이 비율은 이후 Phase 3에서 치료 반응군 vs 비반응군 비교로 확장될 수 있다.

In [ ]:
required_cols = ['sample', 'tam_subtype']
missing_cols = [c for c in required_cols if c not in mac.obs.columns]
print('missing required columns:', missing_cols)

composition = (
    mac.obs
    .groupby(['sample', 'tam_subtype'], observed=False)
    .size()
    .unstack(fill_value=0)
)

composition_pct = composition.div(composition.sum(axis=1), axis=0) * 100

display(composition)
display(composition_pct.round(1))

## 2. Stacked bar plot

Sample별 C1QC+ TAM / SPP1+ TAM / Unknown 비율을 stacked bar plot으로 확인한다.

해석 포인트:
- C1QC+ TAM과 SPP1+ TAM이 함께 관찰되는 sample이 있는지
- Unknown 비율이 과도하게 높은 sample이 있는지
- sample별 TAM infiltration pattern이 균일한지 또는 heterogeneous한지

In [ ]:
composition_pct.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 5),
    colormap='Set2'
)
plt.title('TAM Subtype Composition per Sample')
plt.ylabel('Fraction (%)')
plt.xlabel('Sample')
plt.xticks(rotation=90)
plt.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()


## 3. TAM-rich / TAM-low sample 분류

Unknown 비율이 너무 높으면 해당 sample에서는 C1QC+ TAM / SPP1+ TAM signal이 약하거나, annotation 기준으로 subtype 분리가 충분히 되지 않은 것으로 볼 수 있다.

여기서는 간단히 Unknown 비율 90%를 기준으로 TAM-rich / TAM-low-like sample을 구분해본다.

In [ ]:
unknown_col = 'Unknown'
if unknown_col in composition_pct.columns:
    tam_low_samples = composition_pct[composition_pct[unknown_col] >= 90].index.tolist()
    tam_rich_samples = composition_pct[composition_pct[unknown_col] < 90].index.tolist()

    print('TAM-rich-like samples:', tam_rich_samples)
    print('TAM-low-like / Unknown-dominant samples:', tam_low_samples)

    summary = pd.DataFrame({
        'sample_group': ['TAM-rich-like', 'TAM-low-like'],
        'n_samples': [len(tam_rich_samples), len(tam_low_samples)]
    })
    display(summary)
else:
    print('Unknown column not found in composition table')

## 4. Interpretation note

현재 결과는 sample마다 TAM subtype composition이 다르다는 점을 보여준다.

다만 이 단계만으로 예후나 치료 반응을 직접 말할 수는 없다. 예후/치료 반응과 연결하려면 sample별 임상 정보 또는 treatment response label이 필요하다.

따라서 Phase 2a의 의미는 다음과 같다.

1. C1QC+ TAM / SPP1+ TAM subtype을 GSE127465에서 재현
2. DEG overlap으로 논문 marker signature 재현 확인
3. sample-level TAM composition 분석으로 Phase 3의 치료 반응 비교를 위한 기반 생성

In [ ]:
mac.write('../phase1_scrna/dataset/GSE127465_mac_phase2a_tme.h5ad')
print('저장 완료: ../phase1_scrna/dataset/GSE127465_mac_phase2a_tme.h5ad')